# Experiment: AeroTrack Robustness Under Degraded Conditions

Question: how does the AeroTrack detector behave when aerial footage is degraded by fog, low light, or motion blur?

Success criteria: run the same VisDrone validation images through the original model and progressively degraded variants, then plot detection-count retention by severity. Detection count is a lightweight proxy for mAP when running full validation at every severity is too expensive.

In [ ]:
from __future__ import annotations

import importlib
import subprocess
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "requirements.txt").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

REQUIRED_IMPORTS = {
    "cv2": "opencv-python-headless",
    "matplotlib": "matplotlib",
    "numpy": "numpy",
    "pandas": "pandas",
    "torch": "torch",
    "ultralytics": "ultralytics",
}


def pinned_requirement(package_name: str) -> str:
    requirements_path = PROJECT_ROOT / "requirements.txt"
    normalized_package_name = package_name.lower()
    for line in requirements_path.read_text(encoding="utf-8").splitlines():
        requirement = line.strip()
        if not requirement or requirement.startswith("#"):
            continue
        requirement_name = requirement.split("==", maxsplit=1)[0].split("[", maxsplit=1)[0].lower()
        if requirement_name == normalized_package_name:
            return requirement
    return package_name


missing_requirements = []
for module_name, package_name in REQUIRED_IMPORTS.items():
    try:
        importlib.import_module(module_name)
    except ModuleNotFoundError:
        missing_requirements.append(pinned_requirement(package_name))

if missing_requirements:
    print("Installing missing packages into this Jupyter kernel:", ", ".join(missing_requirements))
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_requirements])

import cv2
import matplotlib.pyplot as plt
import numpy as np

try:
    np.rec
except ModuleNotFoundError:
    import numpy.core.records as np_records
    np.rec = np_records
    sys.modules["numpy.rec"] = np_records

import pandas as pd
from ultralytics import YOLO

try:
    import albumentations as A
    ALBUMENTATIONS_AVAILABLE = True
except Exception as exc:
    A = None
    ALBUMENTATIONS_AVAILABLE = False
    print(
        "Albumentations is unavailable in this kernel; using OpenCV/NumPy fallback "
        f"degradations instead. Import error: {type(exc).__name__}: {exc}"
    )


## Configuration

The notebook assumes it is run from the repository root or from `notebooks/`. Keep constants here so the experiment is easy to rerun with a different checkpoint, severity sweep, or sample count.

In [ ]:
from pathlib import Path

import numpy as np

try:
    np.rec
except ModuleNotFoundError:
    import sys
    import numpy.core.records as np_records
    np.rec = np_records
    sys.modules["numpy.rec"] = np_records


PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "models").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

MODEL_PATH = PROJECT_ROOT / "models" / "aerotrack-detector-demo-v2.pt"
VAL_IMAGE_DIR = PROJECT_ROOT / "data" / "visdrone" / "images" / "val"

SAMPLE_COUNT = 8
IMAGE_SIZE = 640
CONFIDENCE_THRESHOLD = 0.25
IOU_THRESHOLD = 0.45
RANDOM_SEED = 7

SEVERITY_LEVELS = [0, 1, 2, 3]
FOG_COEFFICIENTS = [0.0, 0.15, 0.30, 0.45]
LOW_LIGHT_BRIGHTNESS_LIMITS = [0.0, -0.15, -0.30, -0.45]
MOTION_BLUR_KERNELS = [1, 5, 9, 13]

np.random.seed(RANDOM_SEED)
assert MODEL_PATH.exists(), f"Missing model: {MODEL_PATH}"
assert VAL_IMAGE_DIR.exists(), f"Missing validation images: {VAL_IMAGE_DIR}"


## Operational Degradations

- Fog simulates low-visibility ISR where contrast and object boundaries fade with range.
- Low light simulates dusk, night, or underexposed collection where small objects lose texture.
- Motion blur simulates high-speed platform motion, vibration, or aggressive camera slew.

These are not a replacement for real flight-test data, but they are a fast way to reveal likely field failure modes before deployment.

In [ ]:
def select_representative_images(image_dir: Path, sample_count: int) -> list[Path]:
    image_paths = sorted(image_dir.glob("*.jpg"))
    if len(image_paths) < sample_count:
        raise ValueError(f"Need at least {sample_count} validation images; found {len(image_paths)}")
    step = max(1, len(image_paths) // sample_count)
    return image_paths[::step][:sample_count]


sample_image_paths = select_representative_images(VAL_IMAGE_DIR, SAMPLE_COUNT)
print("Selected validation images:")
for image_path in sample_image_paths:
    print(f"- {image_path.name}")


In [ ]:
model = YOLO(str(MODEL_PATH))
model


## Degradation Helpers

Severity `0` is always the original image. The remaining levels apply stronger perturbations while keeping the same detector thresholds.

In [ ]:
def apply_cv_fog(image: np.ndarray, fog_coefficient: float) -> np.ndarray:
    if fog_coefficient == 0.0:
        return image.copy()
    haze = np.full_like(image, fill_value=255)
    return cv2.addWeighted(image, 1.0 - fog_coefficient, haze, fog_coefficient, 0.0)


def apply_cv_low_light(image: np.ndarray, brightness_limit: float) -> np.ndarray:
    if brightness_limit == 0.0:
        return image.copy()
    beta = brightness_limit * 255.0
    return cv2.convertScaleAbs(image, alpha=1.0, beta=beta)


def apply_cv_motion_blur(image: np.ndarray, kernel_size: int) -> np.ndarray:
    if kernel_size <= 1:
        return image.copy()
    kernel = np.zeros((kernel_size, kernel_size), dtype=np.float32)
    kernel[kernel_size // 2, :] = 1.0 / kernel_size
    return cv2.filter2D(image, ddepth=-1, kernel=kernel)


def make_random_fog(fog_coefficient: float):
    if fog_coefficient == 0.0:
        return None
    if ALBUMENTATIONS_AVAILABLE:
        try:
            return A.RandomFog(
                fog_coef_lower=fog_coefficient,
                fog_coef_upper=fog_coefficient,
                alpha_coef=0.08,
                p=1.0,
            )
        except TypeError:
            return A.RandomFog(fog_coef_range=(fog_coefficient, fog_coefficient), alpha_coef=0.08, p=1.0)
    return lambda image: apply_cv_fog(image, fog_coefficient)


def make_low_light(brightness_limit: float):
    if brightness_limit == 0.0:
        return None
    if ALBUMENTATIONS_AVAILABLE:
        return A.RandomBrightnessContrast(
            brightness_limit=(brightness_limit, brightness_limit),
            contrast_limit=(0.0, 0.0),
            p=1.0,
        )
    return lambda image: apply_cv_low_light(image, brightness_limit)


def make_motion_blur(kernel_size: int):
    if kernel_size <= 1:
        return None
    if ALBUMENTATIONS_AVAILABLE:
        return A.MotionBlur(blur_limit=(kernel_size, kernel_size), p=1.0)
    return lambda image: apply_cv_motion_blur(image, kernel_size)


def apply_transform(image: np.ndarray, transform) -> np.ndarray:
    if transform is None:
        return image.copy()
    if ALBUMENTATIONS_AVAILABLE:
        return transform(image=image)["image"]
    return transform(image)


DEGRADATION_SWEEPS = {
    "fog": list(zip(SEVERITY_LEVELS, FOG_COEFFICIENTS, [make_random_fog(value) for value in FOG_COEFFICIENTS])),
    "low_light": list(zip(SEVERITY_LEVELS, LOW_LIGHT_BRIGHTNESS_LIMITS, [make_low_light(value) for value in LOW_LIGHT_BRIGHTNESS_LIMITS])),
    "motion_blur": list(zip(SEVERITY_LEVELS, MOTION_BLUR_KERNELS, [make_motion_blur(value) for value in MOTION_BLUR_KERNELS])),
}

"Albumentations" if ALBUMENTATIONS_AVAILABLE else "OpenCV/NumPy fallback"


In [ ]:
def read_image_bgr(image_path: Path) -> np.ndarray:
    image = cv2.imread(str(image_path))
    if image is None:
        raise FileNotFoundError(f"Unable to read {image_path}")
    return image


def detection_count(image: np.ndarray) -> int:
    result = model.predict(
        image,
        imgsz=IMAGE_SIZE,
        conf=CONFIDENCE_THRESHOLD,
        iou=IOU_THRESHOLD,
        verbose=False,
    )[0]
    if result.boxes is None:
        return 0
    return len(result.boxes)


## Run Sweep

This cell records detection counts for every image, degradation type, and severity. If runtime becomes an issue, reduce `SAMPLE_COUNT` or the number of severity levels.

In [ ]:
records = []

for image_path in sample_image_paths:
    original_image = read_image_bgr(image_path)
    original_count = detection_count(original_image)

    for degradation_name, sweep in DEGRADATION_SWEEPS.items():
        for severity, parameter_value, transform in sweep:
            degraded_image = apply_transform(original_image, transform)
            degraded_count = detection_count(degraded_image)
            retention = degraded_count / original_count if original_count else np.nan
            records.append(
                {
                    "image": image_path.name,
                    "degradation": degradation_name,
                    "severity": severity,
                    "parameter_value": parameter_value,
                    "original_count": original_count,
                    "degraded_count": degraded_count,
                    "retention": retention,
                }
            )

results_df = pd.DataFrame(records)
results_df.head(12)


In [ ]:
summary_df = (
    results_df.groupby(["degradation", "severity"], as_index=False)
    .agg(
        mean_detection_count=("degraded_count", "mean"),
        mean_retention=("retention", "mean"),
        image_count=("image", "nunique"),
    )
)
summary_df


## Degradation Response Curves

A steep drop means the detector is brittle under that field condition. A flat curve does not prove robustness, but it is a useful first-pass signal before running full mAP by degradation bucket.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4), constrained_layout=True)

for degradation_name, degradation_df in summary_df.groupby("degradation"):
    axes[0].plot(
        degradation_df["severity"],
        degradation_df["mean_detection_count"],
        marker="o",
        label=degradation_name,
    )
    axes[1].plot(
        degradation_df["severity"],
        degradation_df["mean_retention"],
        marker="o",
        label=degradation_name,
    )

axes[0].set_title("Detection count vs. degradation")
axes[0].set_xlabel("Severity")
axes[0].set_ylabel("Mean detections per image")
axes[0].grid(True, alpha=0.3)

axes[1].set_title("Detection retention vs. degradation")
axes[1].set_xlabel("Severity")
axes[1].set_ylabel("Mean retention vs. original")
axes[1].set_ylim(0.0, max(1.05, summary_df["mean_retention"].max() + 0.05))
axes[1].grid(True, alpha=0.3)

for axis in axes:
    axis.legend()


## Visual Sanity Check

Use one image to confirm the perturbations look operationally plausible rather than purely synthetic. The model should be evaluated on degraded images that still resemble camera artifacts, not arbitrary corruptions.

In [ ]:
preview_image = read_image_bgr(sample_image_paths[0])
preview_specs = [
    ("original", preview_image),
    ("fog", apply_transform(preview_image, DEGRADATION_SWEEPS["fog"][-1][2])),
    ("low_light", apply_transform(preview_image, DEGRADATION_SWEEPS["low_light"][-1][2])),
    ("motion_blur", apply_transform(preview_image, DEGRADATION_SWEEPS["motion_blur"][-1][2])),
]

fig, axes = plt.subplots(1, len(preview_specs), figsize=(14, 4), constrained_layout=True)
for axis, (title, image) in zip(axes, preview_specs):
    axis.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    axis.set_title(title)
    axis.axis("off")


## Interpretation Prompts

- Which degradation drops detection count fastest, and does that line up with the intended operating envelope?
- Are failures concentrated in small-object classes such as bicycle, motor, tricycle, or awning-tricycle?
- Should the next training pass include augmentation for the most brittle condition?
- Before using these results publicly, run full mAP by degradation bucket or manually inspect false positives and false negatives on the worst cases.